# LangGraph Agent with AgentCore Deployment

This notebook demonstrates how to build a ReAct-style agent using **LangGraph** and **AWS Bedrock**, then deploy it as a managed service on **Amazon Bedrock AgentCore Runtime**.

You'll learn how to:
- Configure and connect to Claude models via Bedrock
- Define custom tools that the agent can use
- Build a graph-based agent that reasons and acts
- Test the agent with various queries
- Package and deploy the agent to AgentCore Runtime
- Invoke the deployed agent via CLI and boto3

**Prerequisites:** Ensure your model is configured in `../CONFIG.txt` before running.

## 1. Configuration

Load the model ID and AWS region from `CONFIG.txt`. For cross-region inference profiles (MODEL_ID starting with `us.`), we also derive a `BASE_MODEL_ID` that some LangChain components require.

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "langchain-mcp-adapters",
    "mcp",
    "httpx",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install missing packages
%pip install langgraph>=1.0.6 -q

## 2. Imports

Import the required libraries for building our agent:
- **LangChain AWS**: Provides the `ChatBedrockConverse` class for Bedrock integration
- **LangGraph**: Framework for building stateful, graph-based agents
- **LangChain Core**: Message types and tool decorators

In [ ]:
from typing import Literal
from datetime import datetime

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

print("Imports successful!")

## 3. Define Tools

Tools are functions that the agent can call to perform actions or retrieve information. We use the `@tool` decorator to register them with LangChain.

Here we define two simple tools:
- **get_current_time**: Returns the current date and time
- **add_numbers**: Adds two integers together

The agent will automatically decide when to use these tools based on the user's question.

In [ ]:
@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


tools = [get_current_time, add_numbers]
print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

## 4. Initialize the LLM

Create a connection to Claude via AWS Bedrock using `ChatBedrockConverse`. This class uses the Bedrock Converse API, which provides a unified interface for all supported models.

Key configuration:
- **model**: The model ID from CONFIG.txt (e.g., `us.anthropic.claude-sonnet-4-5-20250929-v1:0`)
- **region_name**: AWS region where Bedrock is available
- **temperature**: Controls randomness (0 = deterministic, 1 = creative)
- **base_model_id**: Required for cross-region inference profiles to identify the underlying model

After initialization, we bind our tools to the LLM so it knows what actions are available.

In [ ]:
# Build LLM config - add base_model_id for Claude inference profiles
llm_kwargs = {
    "model": MODEL_ID,
    "region_name": REGION,
    "temperature": 0,
}

if BASE_MODEL_ID:
    llm_kwargs["base_model_id"] = BASE_MODEL_ID

llm = ChatBedrockConverse(**llm_kwargs)

# Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

print(f"LLM initialized with {MODEL_ID}!")

## 5. Build the LangGraph Agent

LangGraph allows us to build agents as **directed graphs** where:
- **Nodes** are functions that process state (e.g., call the LLM, execute tools)
- **Edges** define the flow between nodes
- **Conditional edges** route based on the current state

Our agent follows the **ReAct pattern** (Reason + Act):
1. The `agent` node calls the LLM to decide what to do
2. If the LLM requests tool calls, route to the `tools` node
3. The `tools` node executes the requested tools
4. Return to `agent` to process results and continue reasoning
5. When no more tools are needed, end the conversation

In [ ]:
def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """Determine whether to continue to tools or end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


def call_model(state: MessagesState):
    """Call the LLM."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# Build the graph
graph = StateGraph(MessagesState)

# Add nodes
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(tools))

# Add edges
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")

# Compile
agent = graph.compile()

print("Agent graph compiled successfully!")

## 6. Visualize the Graph (Optional)

LangGraph can render the agent's graph structure as a diagram. This helps visualize how messages flow between nodes and understand the agent's decision-making process.

**Note:** Requires `graphviz` to be installed. If not available, a text representation is shown instead.

In [ ]:
# Display the graph structure (requires graphviz)
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visualization not available: {e}")
    print("\nGraph structure:")
    print("  START -> agent -> (tools -> agent) | END")

## 7. Run the Agent

Now let's test the agent! The `run_agent` function:
1. Takes a question as input
2. Wraps it with a system message defining the assistant's behavior
3. Invokes the agent graph
4. Returns the final response

The agent will automatically use tools when needed to answer the question.

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"Question: {question}")
    print("-" * 50)

    result = agent.invoke({
        "messages": [
            SystemMessage(content="You are a helpful assistant. Use tools when needed."),
            HumanMessage(content=question),
        ]
    })

    final_message = result["messages"][-1]
    print(f"\nResponse:\n{final_message.content}")
    return result

In [ ]:
# Test: Get current time
result = run_agent("What is the current time?")

In [ ]:
# Test: Math calculation
result = run_agent("What is 42 + 17?")

In [ ]:
# Test: Multiple tools
result = run_agent("What time is it and what is 100 + 200?")

## 8. Query with Sample Financial Data

This section demonstrates how to provide context to the agent by loading sample SEC 10-K filing data and asking questions about it.

This pattern is useful for:
- **RAG (Retrieval-Augmented Generation)**: Injecting retrieved documents into prompts
- **In-context learning**: Providing examples or data for the agent to reference
- **Domain-specific Q&A**: Answering questions based on financial filing information

In [ ]:
# Load sample SEC financial filing data
from load_sample_data import load_financial_data, print_info

financial_text = load_financial_data()
print_info(financial_text)

In [ ]:
# Ask the agent questions about the financial data
def ask_about_data(question: str, context: str):
    """Ask the agent a question with context."""
    prompt = f"""Based on this SEC 10-K filing information:

{context}

Question: {question}"""
    return run_agent(prompt)

# Sample question about companies and products
ask_about_data("What companies are mentioned and what are their key products?", financial_text)

In [ ]:
# Another sample question about risk factors
ask_about_data("What risk factors are mentioned and which companies face them?", financial_text)

---

## 9. Deploy to AgentCore Runtime

Now that the agent works locally, let's deploy it as a **managed service** on Amazon Bedrock AgentCore Runtime. This packages the agent code and deploys it using `direct_code_deploy` — no Docker required.

**Architecture:**
```
User → AgentCore Runtime → ReAct Agent (Claude) → Tools
```

The deployment process:
1. Create a self-contained directory with the agent code, dependencies, and config
2. Use the `agentcore` CLI to deploy
3. Invoke the deployed agent via CLI or boto3

In [ ]:
%pip install bedrock-agentcore-starter-toolkit>=0.3.3 bedrock-agentcore>=1.4.7 pyyaml -q

## 10. Create the Deployment Package

AgentCore's `direct_code_deploy` packages an entire directory and uploads it. We'll create a self-contained directory with:
- `agent.py` — the agent code wrapped in a `BedrockAgentCoreApp` handler
- `pyproject.toml` — Python dependencies

In [ ]:
!mkdir -p agentcore_deploy
print("Created agentcore_deploy/ directory")

In [ ]:
%%writefile agentcore_deploy/agent.py
#!/usr/bin/env python3
"""
Basic ReAct Agent - AgentCore Runtime Deployment

A simple ReAct agent with time and math tools, deployed to AgentCore Runtime.
"""

import logging
import os
from datetime import datetime

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-20250514-v1:0")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

SYSTEM_PROMPT = "You are a helpful assistant. Use the available tools when needed to answer questions."


@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


def get_llm():
    return init_chat_model(
        MODEL_ID,
        model_provider="bedrock_converse",
        region_name=AWS_REGION,
        temperature=0,
    )


@app.entrypoint
async def invoke(payload: dict = None):
    """AgentCore Runtime handler."""
    if payload is None:
        payload = {}

    prompt = (
        payload.get("prompt")
        or payload.get("message")
        or payload.get("query")
        or payload.get("input")
    )

    if not prompt:
        yield {"type": "error", "error": "No prompt provided. Include 'prompt' in your request."}
        return

    logger.info(f"Query: {prompt[:100]}...")

    try:
        llm = get_llm()
        tools = [get_current_time, add_numbers]
        agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

        result = await agent.ainvoke({"messages": [("human", prompt)]})

        messages = result.get("messages", [])
        if messages and hasattr(messages[-1], "content"):
            response_text = messages[-1].content
        else:
            response_text = "No response from agent"

        yield {"type": "chunk", "data": response_text}
        yield {"type": "complete"}

        logger.info("Request completed successfully")

    except Exception as e:
        logger.error(f"Error: {e}", exc_info=True)
        yield {"type": "error", "error": f"Error processing request: {str(e)}"}


if __name__ == "__main__":
    app.run(port=8080)

In [ ]:
%%writefile agentcore_deploy/pyproject.toml
[project]
name = "basic-react-agent"
version = "0.1.0"
description = "Basic ReAct agent with time and math tools"
requires-python = ">=3.10"
dependencies = [
    "langchain>=1.2.0",
    "langgraph>=1.0.6",
    "langchain-aws>=1.2.0",
    "boto3>=1.42.0",
    "bedrock-agentcore>=1.4.7",
]

In [ ]:
!ls -la agentcore_deploy/

## 11. Configure and Deploy

AgentCore needs a `.bedrock_agentcore.yaml` config file that specifies the entrypoint, deployment type, and AWS settings. We generate this programmatically to avoid interactive CLI prompts.

The IAM execution role and S3 bucket are auto-created by the toolkit.

In [ ]:
import boto3
import yaml

# Get AWS account ID
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

# Build absolute paths for AgentCore config
agent_dir = os.path.abspath("agentcore_deploy")
entrypoint = os.path.join(agent_dir, "agent.py")

AGENT_NAME = "basic_react_agent"

config = {
    "default_agent": AGENT_NAME,
    "agents": {
        AGENT_NAME: {
            "name": AGENT_NAME,
            "language": "python",
            "entrypoint": entrypoint,
            "deployment_type": "direct_code_deploy",
            "runtime_type": "PYTHON_3_13",
            "platform": "linux/arm64",
            "source_path": agent_dir,
            "aws": {
                "account": account_id,
                "region": REGION,
                "execution_role_auto_create": True,
                "ecr_auto_create": False,
                "s3_auto_create": True,
                "network_configuration": {
                    "network_mode": "PUBLIC",
                },
                "protocol_configuration": {
                    "server_protocol": "HTTP",
                },
                "observability": {
                    "enabled": True,
                },
            },
        }
    },
}

config_path = os.path.join(agent_dir, ".bedrock_agentcore.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Wrote {config_path}")
print(f"  Agent:      {AGENT_NAME}")
print(f"  Account:    {account_id}")
print(f"  Region:     {REGION}")
print(f"  Deploy:     direct_code_deploy")

In [ ]:
# Install zip if not available (needed by direct_code_deploy)
!which zip || (sudo apt-get update -qq && sudo apt-get install -y -qq zip)

# Deploy
!cd agentcore_deploy && agentcore deploy

## 12. Invoke the Deployed Agent

Once deployed, invoke the agent via the `agentcore` CLI or programmatically with boto3.

In [ ]:
# Invoke via CLI
!cd agentcore_deploy && agentcore invoke '{"prompt": "What time is it and what is 42 + 17?"}'

### Invoke via boto3

For programmatic access, use the `bedrock-agentcore` boto3 client. First, extract the agent ARN from the deployment config:

In [ ]:
import json
import uuid
import yaml
from botocore.config import Config

# Read the agent ARN from the deployment config
with open("agentcore_deploy/.bedrock_agentcore.yaml") as f:
    deploy_config = yaml.safe_load(f)

default_agent = deploy_config["default_agent"]
agent_config = deploy_config["agents"][default_agent]
AGENT_ARN = agent_config["bedrock_agentcore"]["agent_arn"]
AGENT_REGION = agent_config["aws"]["region"]

print(f"Agent:  {default_agent}")
print(f"ARN:    {AGENT_ARN}")
print(f"Region: {AGENT_REGION}")

In [ ]:
# AgentCore agents may take time to respond, so increase the read timeout
agentcore_config = Config(read_timeout=300)


def invoke_agent(prompt: str):
    """Invoke the deployed agent via boto3."""
    client = boto3.client(
        "bedrock-agentcore",
        region_name=AGENT_REGION,
        config=agentcore_config,
    )

    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )

    content = "".join(
        chunk.decode("utf-8") for chunk in response.get("response", [])
    )

    try:
        return json.loads(content)
    except (json.JSONDecodeError, ValueError):
        return content


result = invoke_agent("What is 100 + 200?")
print(result)

## 13. Cleanup

When you're done, remove the agent from AgentCore Runtime. This deletes the deployed runtime but keeps your local files.

In [ ]:
# Uncomment the line below to destroy the deployed agent
# !cd agentcore_deploy && agentcore destroy